In [ ]:
import json
from pathlib import Path

import torch
import torch.nn as nn


# ============================================================
# 設定
# ============================================================

MODEL_DIR = Path("char_bilstm_detector")

MODEL_PATH = MODEL_DIR / "model.pt"
VOCAB_PATH = MODEL_DIR / "char_vocab.json"
CONFIG_PATH = MODEL_DIR / "config.json"

PAD_ID = 0
UNK_ID = 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# Char BiLSTM + Sentence BiLSTM
# ============================================================

class CharBiLSTMTokenClassifier(nn.Module):

    def __init__(
        self,
        char_vocab_size,
        char_embed_dim,
        char_hidden_dim,
        sent_hidden_dim,
        num_labels
    ):

        super().__init__()

        # ----------------------------------------------------
        # Character Embedding
        # ----------------------------------------------------

        self.char_embedding = nn.Embedding(
            num_embeddings=char_vocab_size,
            embedding_dim=char_embed_dim,
            padding_idx=PAD_ID
        )

        # ----------------------------------------------------
        # Char BiLSTM
        #
        # 一個 Token：
        #
        # ORD90550145
        #
        # ↓
        #
        # O R D 9 0 5 5 0 1 4 5
        #
        # ↓
        #
        # Character Embedding
        #
        # ↓
        #
        # Char BiLSTM
        # ----------------------------------------------------

        self.char_lstm = nn.LSTM(
            input_size=char_embed_dim,
            hidden_size=char_hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        # ----------------------------------------------------
        # Sentence BiLSTM
        #
        # Token 1
        # Token 2
        # Token 3
        # ...
        #
        # ↓
        #
        # Sentence BiLSTM
        # ----------------------------------------------------

        self.sent_lstm = nn.LSTM(
            input_size=char_hidden_dim * 2,
            hidden_size=sent_hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        # ----------------------------------------------------
        # Classifier
        #
        # → 0 / 1
        # ----------------------------------------------------

        self.classifier = nn.Linear(
            sent_hidden_dim * 2,
            num_labels
        )

    def forward(
        self,
        char_ids,
        char_lengths,
        token_mask
    ):

        batch_size = char_ids.size(0)
        sentence_length = char_ids.size(1)
        char_length = char_ids.size(2)

        # ====================================================
        # [B, S, C]
        #
        # ↓
        #
        # [B*S, C]
        # ====================================================

        flat_char_ids = char_ids.reshape(
            batch_size * sentence_length,
            char_length
        )

        flat_char_lengths = char_lengths.reshape(
            batch_size * sentence_length
        )

        # ====================================================
        # Character Embedding
        # ====================================================

        embedded = self.char_embedding(
            flat_char_ids
        )

        # ====================================================
        # Char BiLSTM
        # ====================================================

        safe_lengths = flat_char_lengths.clamp(
            min=1
        ).cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            safe_lengths,
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.char_lstm(
            packed
        )

        # hidden:
        #
        # [2, B*S, Hidden]
        #
        # 第一個 = forward
        # 第二個 = backward

        char_forward = hidden[-2]
        char_backward = hidden[-1]

        token_vectors = torch.cat(
            [
                char_forward,
                char_backward
            ],
            dim=1
        )

        # ====================================================
        # [B*S, Hidden*2]
        #
        # ↓
        #
        # [B, S, Hidden*2]
        # ====================================================

        token_vectors = token_vectors.reshape(
            batch_size,
            sentence_length,
            -1
        )

        # Padding Token 歸零

        token_vectors = token_vectors.masked_fill(
            ~token_mask.unsqueeze(-1),
            0
        )

        # ====================================================
        # Sentence BiLSTM
        # ====================================================

        sentence_lengths = token_mask.sum(
            dim=1
        ).cpu()

        packed_sentence = nn.utils.rnn.pack_padded_sequence(
            token_vectors,
            sentence_lengths,
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.sent_lstm(
            packed_sentence
        )

        sentence_output, _ = nn.utils.rnn.pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=sentence_length
        )

        # ====================================================
        # Classifier
        # ====================================================

        logits = self.classifier(
            sentence_output
        )

        return logits


# ============================================================
# 載入模型
# ============================================================

def load_model():

    print("正在載入模型...")

    # --------------------------------------------------------
    # 檢查檔案
    # --------------------------------------------------------

    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            f"找不到模型：{MODEL_PATH}\n"
            f"請先執行 train.py"
        )

    if not VOCAB_PATH.exists():
        raise FileNotFoundError(
            f"找不到 Vocabulary：{VOCAB_PATH}\n"
            f"請先執行 train.py"
        )

    if not CONFIG_PATH.exists():
        raise FileNotFoundError(
            f"找不到設定檔：{CONFIG_PATH}\n"
            f"請先執行 train.py"
        )

    # --------------------------------------------------------
    # Vocabulary
    # --------------------------------------------------------

    with open(
        VOCAB_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        char_vocab = json.load(f)

    # --------------------------------------------------------
    # Config
    # --------------------------------------------------------

    with open(
        CONFIG_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        config = json.load(f)

    # --------------------------------------------------------
    # 建立模型
    # --------------------------------------------------------

    model = CharBiLSTMTokenClassifier(
        char_vocab_size=len(char_vocab),
        char_embed_dim=config["char_embed_dim"],
        char_hidden_dim=config["char_hidden_dim"],
        sent_hidden_dim=config["sent_hidden_dim"],
        num_labels=config["num_labels"]
    )

    # --------------------------------------------------------
    # 載入權重
    # --------------------------------------------------------

    state_dict = torch.load(
        MODEL_PATH,
        map_location=DEVICE
    )

    model.load_state_dict(
        state_dict
    )

    model.to(DEVICE)

    model.eval()

    print(f"Vocabulary：{len(char_vocab)} 個字元")
    print(f"Device：{DEVICE}")
    print("模型載入完成\n")

    return model, char_vocab, config


# ============================================================
# 將 Token 轉成 Character ID
# ============================================================

def token_to_char_ids(
    token,
    char_vocab,
    max_char_length
):

    # --------------------------------------------------------
    # Token
    #
    # ORD90550145
    #
    # ↓
    #
    # O R D 9 0 5 5 0 1 4 5
    # --------------------------------------------------------

    chars = list(token)

    # 限制最大長度

    chars = chars[:max_char_length]

    if not chars:
        return [UNK_ID]

    ids = []

    for char in chars:

        # 如果訓練時沒出現過這個字元
        # 使用 UNK

        char_id = char_vocab.get(
            char,
            UNK_ID
        )

        ids.append(char_id)

    return ids


# ============================================================
# 建立 Prediction Tensor
# ============================================================

def prepare_input(
    tokens,
    char_vocab,
    max_char_length
):

    if not tokens:
        raise ValueError(
            "tokens 不可以是空的"
        )

    # 每個 Token 的 Character IDs

    char_sequences = []

    for token in tokens:

        ids = token_to_char_ids(
            token,
            char_vocab,
            max_char_length
        )

        char_sequences.append(ids)

    # Sentence 長度

    sentence_length = len(tokens)

    # 找最長 Token

    max_token_length = max(
        len(x)
        for x in char_sequences
    )

    # --------------------------------------------------------
    # [1, Sentence Length, Character Length]
    # --------------------------------------------------------

    char_tensor = torch.full(
        (
            1,
            sentence_length,
            max_token_length
        ),
        PAD_ID,
        dtype=torch.long
    )

    # --------------------------------------------------------
    # Token Mask
    # --------------------------------------------------------

    token_mask = torch.ones(
        (
            1,
            sentence_length
        ),
        dtype=torch.bool
    )

    # --------------------------------------------------------
    # 每個 Token 的字元長度
    # --------------------------------------------------------

    char_lengths = torch.zeros(
        (
            1,
            sentence_length
        ),
        dtype=torch.long
    )

    # --------------------------------------------------------
    # 填入資料
    # --------------------------------------------------------

    for i, ids in enumerate(char_sequences):

        length = len(ids)

        char_tensor[
            0,
            i,
            :length
        ] = torch.tensor(
            ids,
            dtype=torch.long
        )

        char_lengths[
            0,
            i
        ] = length

    return (
        char_tensor,
        token_mask,
        char_lengths
    )


# ============================================================
# 預測
# ============================================================

@torch.no_grad()
def predict(
    model,
    char_vocab,
    config,
    tokens
):

    # --------------------------------------------------------
    # 建立輸入
    # --------------------------------------------------------

    (
        char_ids,
        token_mask,
        char_lengths
    ) = prepare_input(
        tokens,
        char_vocab,
        config["max_char_length"]
    )

    char_ids = char_ids.to(DEVICE)
    token_mask = token_mask.to(DEVICE)
    char_lengths = char_lengths.to(DEVICE)

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    logits = model(
        char_ids,
        char_lengths,
        token_mask
    )

    # --------------------------------------------------------
    # Softmax
    #
    # logits
    # ↓
    #
    # P(0)
    # P(1)
    # --------------------------------------------------------

    probabilities = torch.softmax(
        logits,
        dim=-1
    )

    predictions = torch.argmax(
        probabilities,
        dim=-1
    )

    results = []

    for i, token in enumerate(tokens):

        p0 = probabilities[
            0,
            i,
            0
        ].item()

        p1 = probabilities[
            0,
            i,
            1
        ].item()

        prediction = predictions[
            0,
            i
        ].item()

        results.append({
            "token": token,
            "probability_0": p0,
            "probability_1": p1,
            "prediction": prediction
        })

    return results


# ============================================================
# 顯示結果
# ============================================================

def print_results(results):

    print()
    print("=" * 78)
    print(
        f"{'Token':<25}"
        f"{'P(0)':>12}"
        f"{'P(1)':>12}"
        f"{'Prediction':>15}"
    )
    print("=" * 78)

    for result in results:

        token = result["token"]

        # Token 太長時避免破壞表格

        if len(token) > 23:
            token = token[:20] + "..."

        p0 = result["probability_0"]
        p1 = result["probability_1"]
        prediction = result["prediction"]

        print(
            f"{token:<25}"
            f"{p0:>11.2%}"
            f"{p1:>11.2%}"
            f"{prediction:>15}"
        )

    print("=" * 78)

    sensitive_tokens = [
        result["token"]
        for result in results
        if result["prediction"] == 1
    ]

    if sensitive_tokens:

        print("\n偵測到的重要資訊：")

        for token in sensitive_tokens:

            result = next(
                x for x in results
                if x["token"] == token
            )

            print(
                f"  [1] {token} "
                f"({result['probability_1']:.2%})"
            )

    else:

        print("\n沒有偵測到重要資訊。")


# ============================================================
# 互動模式
# ============================================================

def interactive_mode(
    model,
    char_vocab,
    config
):

    print("=" * 78)
    print("Sensitive Token Detector")
    print("=" * 78)

    print()
    print("輸入一整句文字進行分析。")
    print("例如：")
    print()
    print(
        "Please include ORD73829104 "
        "in the transfer memo"
    )
    print()
    print("輸入 exit 離開。")
    print()

    while True:

        try:

            text = input("> ")

        except (
            KeyboardInterrupt,
            EOFError
        ):

            print()
            break

        text = text.strip()

        if not text:
            continue

        if text.lower() in (
            "exit",
            "quit",
            "q"
        ):
            break

        # ----------------------------------------------------
        # 你的資料本身就是 Token List
        #
        # 這裡使用最簡單的空白切分
        # ----------------------------------------------------

        tokens = text.split()

        if not tokens:
            continue

        results = predict(
            model,
            char_vocab,
            config,
            tokens
        )

        print_results(
            results
        )


# ============================================================
# 主程式
# ============================================================

def main():

    model, char_vocab, config = load_model()

    interactive_mode(
        model,
        char_vocab,
        config
    )


if __name__ == "__main__":
    main()

正在載入模型...
Vocabulary：72 個字元
Device：cpu
模型載入完成

Sensitive Token Detector

輸入一整句文字進行分析。
例如：

Please include ORD73829104 in the transfer memo

輸入 exit 離開。


Token                            P(0)        P(1)     Prediction
train.jsonl                   99.77%      0.23%              0
↓                             99.49%      0.51%              0
自動建立                          97.96%      2.04%              0
Character                     76.27%     23.73%              0
Vocabulary                     3.73%     96.27%              1
↓                             98.85%      1.15%              0
Character                     94.62%      5.38%              0
Embedding                     12.30%     87.70%              1
↓                             98.28%      1.72%              0
Char                          98.78%      1.22%              0
BiLSTM                         1.54%     98.46%              1
↓                             87.35%     12.65%              0
每個                       